In [1]:
import os
import docker 
import subprocess
import requests
from docker.errors import DockerException

client = docker.from_env()## used to make connection to the docker daemon running on the machine
print(client.version())

{'Platform': {'Name': 'Docker Desktop 4.66.0 (222299)'}, 'Version': '29.3.0', 'ApiVersion': '1.54', 'MinAPIVersion': '1.40', 'Os': 'linux', 'Arch': 'amd64', 'Components': [{'Name': 'Engine', 'Version': '29.3.0', 'Details': {'ApiVersion': '1.54', 'Arch': 'amd64', 'BuildTime': '2026-03-05T14:25:43.000000000+00:00', 'Experimental': 'false', 'GitCommit': '83bca51', 'GoVersion': 'go1.25.7', 'KernelVersion': '6.6.87.2-microsoft-standard-WSL2', 'MinAPIVersion': '1.40', 'Os': 'linux'}}, {'Name': 'containerd', 'Version': 'v2.2.1', 'Details': {'GitCommit': 'dea7da592f5d1d2b7755e3a161be07f43fad8f75'}}, {'Name': 'runc', 'Version': '1.3.4', 'Details': {'GitCommit': 'v1.3.4-0-gd6d73eb8'}}, {'Name': 'docker-init', 'Version': '0.19.0', 'Details': {'GitCommit': 'de40ad0'}}], 'GitCommit': '83bca51', 'GoVersion': 'go1.25.7', 'KernelVersion': '6.6.87.2-microsoft-standard-WSL2', 'BuildTime': '2026-03-05T14:25:43.000000000+00:00'}


In [3]:
!pip install pywin32


[notice] A new release of pip is available: 23.2.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:

result = subprocess.run(['echo', 'Hello from Aditya'],shell = True, capture_output=True, text=True)
print(result.stdout)

"Hello from Aditya"



In [5]:
result = subprocess.run(["python","--version"],capture_output = True,text = True)
print(result.stdout)

Python 3.12.0



In [6]:
result= subprocess.run(
    ["g++","code.cpp","-o","code"],
    capture_output=True,
    text=True)

if result.returncode != 0:
    print("Compilation ERROR!!")
    print("Return code: ",result.returncode)
    print("STDOUT:")
    print(result.stdout)

    print("STDERR:")
    print(result.stderr)
else: 
    # run executable 
    res = subprocess.run(["code"],capture_output=True,text=True).stdout
    print(res)
    


FileNotFoundError: [WinError 2] The system cannot find the file specified

In [7]:
logs = client.containers.run(
    image="alpine",
    command="echo hello codeforge rce!",
    remove=True
)
print(logs.decode())

hello codeforge rce!



In [ ]:
def execute_code():
    host_path = os.path.abspath("workspace")
    container = None
    try:
        container = client.containers.run(
        image="codeforge-cpp",
        command='sh -c "g++ /app/code.cpp -o /app/out && /app/out"',
        volumes={
            host_path:{
                "bind":"/app",
                 "mode":"rw"
            }
        },
        detach=True,
        network_disabled=True,
        mem_limit="128m",
        pids_limit=64
        )
        result = container.wait(timeout=5)
        logs = container.logs().decode()
        container.reload()
        state = container.attrs["State"]
        if(state["OOMKilled"]):
            return {
        "status": "MLE",
        "logs": "Memory Limit Exceeded",
        "status_code": 137
        }
        return {
            "status":"SUCCESS " if result["StatusCode"]==0 else "ERROR",
            "logs":logs,
            "status_code":result["StatusCode"]
        }

    except requests.exceptions.ConnectionError:
        if container:
            try:
                 container.kill()
            except:
                pass
        return {
            "status": "TLE",
            "logs": "Time Limit Exceeded (5 seconds)",
            "status_code": -1
        }
    except Exception as e:

        return {
            "status": "DOCKER_ERROR",
            "logs": str(e),
            "status_code": -1
        }
    finally:
        if container:
            try:
                container.remove(force=True)
            except:
                pass
        

execute_code()



{'Status': 'exited', 'Running': False, 'Paused': False, 'Restarting': False, 'OOMKilled': True, 'Dead': False, 'Pid': 0, 'ExitCode': 137, 'Error': '', 'StartedAt': '2026-06-06T14:55:57.122359582Z', 'FinishedAt': '2026-06-06T14:55:58.263717837Z'}


{'status': 'ERROR', 'logs': 'Killed\n', 'status_code': 137}